# Notebook 03 — Nonlinearities and the Bias-Variance Trade-Off in Control Selection

Notebooks 01 and 02 assumed a linear wage equation. This notebook addresses two complications that arise in practice:

1. **The outcome transformation:** Why log(wage) instead of wage? The choice is not cosmetic — it changes the estimand and whether the treatment response is linear.

2. **Nonlinear treatment response:** Work experience has a concave effect on wages (diminishing returns). Ignoring this misspecifies the model. The Mincer (1974) quadratic term is the standard fix, and FWL applies to it directly.

3. **Control selection trade-offs:** Not all controls are equally desirable.
   - **Neutral controls** (variables that predict wages but not schooling) reduce variance without introducing bias — include them.
   - **Noise-inducing controls** (variables that predict schooling but have weak direct effects on wages) inflate the standard error on the schooling coefficient — think carefully before including them.
   - **Confounders** (variables that predict both) must be included to remove bias, even if they also inflate the SE.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import wooldridge as woo

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

df = woo.dataWoo('wage2').copy()
df['log_wage'] = np.log(df['wage'])
df = df.dropna(subset=['IQ']).copy()
print(f"N = {len(df)}")

## 1. Why Log Wages?

The choice between `wage` and `log(wage)` as the outcome is not neutral. It determines the *interpretation* of the treatment coefficient and whether the linear model is well-specified.

- **Level model:** β measures the dollar increase in monthly wages per year of schooling — assumes a constant $X raise regardless of whether you're earning $500 or $3,000/month. 
- **Log model:** β measures the *proportional* increase — a constant percentage return regardless of income level. This is the Mincer (1974) specification and matches how we talk about wage returns in practice.

There is also a practical reason: wages are right-skewed, violating the OLS homoskedasticity assumption. Log wages are approximately normally distributed, improving model fit and inference.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['wage'], bins=50, color='#7f7f7f', edgecolor='white', alpha=0.85)
axes[0].axvline(df['wage'].mean(), color='#d62728', linewidth=2, linestyle='--', label=f'Mean = ${df["wage"].mean():.0f}')
axes[0].axvline(df['wage'].median(), color='#1f77b4', linewidth=2, linestyle='--', label=f'Median = ${df["wage"].median():.0f}')
axes[0].set_xlabel('Monthly Wage ($)', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
axes[0].set_title('Raw Wage: Right-Skewed\n(mean ≠ median, heteroskedastic)', fontsize=10)
axes[0].legend(fontsize=9)

axes[1].hist(df['log_wage'], bins=50, color='#1f77b4', edgecolor='white', alpha=0.85)
axes[1].axvline(df['log_wage'].mean(), color='#d62728', linewidth=2, linestyle='--', label=f'Mean = {df["log_wage"].mean():.2f}')
axes[1].axvline(df['log_wage'].median(), color='navy', linewidth=2, linestyle='--', label=f'Median = {df["log_wage"].median():.2f}')
axes[1].set_xlabel('Log Monthly Wage', fontsize=11)
axes[1].set_ylabel('Count', fontsize=11)
axes[1].set_title('Log Wage: Approximately Normal\n(better OLS properties)', fontsize=10)
axes[1].legend(fontsize=9)

plt.suptitle('Outcome Transformation: Level vs. Log', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/03_wage_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

# Compare level vs. log estimates side by side
level_model = smf.ols('wage ~ educ + IQ + exper + I(exper**2) + tenure + married + black + south + urban', data=df).fit()
log_model   = smf.ols('log_wage ~ educ + IQ + exper + I(exper**2) + tenure + married + black + south + urban', data=df).fit()

print("Level model β_educ:", level_model.params['educ'].round(3),
      f"→ each year of schooling raises monthly wages by ${level_model.params['educ']:.2f}")
print("Log model β_educ:  ", log_model.params['educ'].round(4),
      f"→ each year of schooling raises wages by ~{log_model.params['educ']*100:.1f}%")

## 2. The Mincer Quadratic: Experience as a Nonlinear Treatment

In the Mincer (1974) earnings model, experience enters as a quadratic term: $\text{exper} + \text{exper}^2$. The rationale is that returns to experience are concave — early years of experience build skills rapidly; additional years eventually add less.

The peak of the experience-wage profile is at exper* = −β_exper / (2 × β_exper²). Beyond this, wages begin to decline in the model (in practice, selection effects complicate this).

FWL applies to nonlinear treatments: to apply it here, we simply apply the nonlinear transformation first, then residualize the transformed treatment on the controls.

In [ ]:
# Fit linear vs. quadratic experience models
m_linear = smf.ols('log_wage ~ educ + IQ + exper + tenure + married + black + south + urban', data=df).fit()
m_quad   = smf.ols('log_wage ~ educ + IQ + exper + I(exper**2) + tenure + married + black + south + urban', data=df).fit()

# Show experience profile
exper_vals = np.linspace(1, 23, 200)
b_lin  = m_linear.params['exper']
b_q1   = m_quad.params['exper']
b_q2   = m_quad.params['I(exper ** 2)']
peak   = -b_q1 / (2 * b_q2)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: scatter of experience vs. log wages
bins = pd.cut(df['exper'], bins=range(0, 25, 2))
avg  = df.groupby(bins)['log_wage'].mean()
cnt  = df.groupby(bins)['log_wage'].count()
midpoints = [b.mid for b in avg.index]
axes[0].scatter(midpoints, avg.values, s=cnt.values*3, color='#1f77b4', alpha=0.8, label='Mean log wage (bubble = N)')
axes[0].plot(exper_vals, m_linear.params['Intercept'] + b_lin * exper_vals + m_linear.params['educ'] * df['educ'].mean() + m_linear.params['IQ'] * df['IQ'].mean(),
             color='#d62728', linewidth=2, linestyle='--', label='Linear fit')
axes[0].plot(exper_vals, m_quad.params['Intercept'] + b_q1 * exper_vals + b_q2 * exper_vals**2 + m_quad.params['educ'] * df['educ'].mean() + m_quad.params['IQ'] * df['IQ'].mean(),
             color='#2ca02c', linewidth=2.5, label='Quadratic fit (Mincer)')
axes[0].axvline(peak, color='#2ca02c', linewidth=1.2, linestyle=':', alpha=0.7)
axes[0].text(peak + 0.3, avg.min() + 0.05, f'Peak ≈ {peak:.0f} yrs', fontsize=9, color='#2ca02c')
axes[0].set_xlabel('Years of Experience', fontsize=11)
axes[0].set_ylabel('Mean Log Monthly Wage', fontsize=11)
axes[0].set_title('Experience-Wage Profile: Concave\n(quadratic is better specified)', fontsize=10)
axes[0].legend(fontsize=9)

# Right: model comparison
print(f"Linear experience model  — β_educ: {m_linear.params['educ']:.5f}, R²: {m_linear.rsquared:.4f}")
print(f"Quadratic experience model β_educ: {m_quad.params['educ']:.5f}, R²: {m_quad.rsquared:.4f}")
print(f"Experience peak: {peak:.1f} years")
print()
print("Mincer interpretation of quadratic:")
print(f"  At exper = 5:  Δlog_wage/Δexper = {b_q1 + 2*b_q2*5:.4f}")
print(f"  At exper = 10: Δlog_wage/Δexper = {b_q1 + 2*b_q2*10:.4f}")
print(f"  At exper = 15: Δlog_wage/Δexper = {b_q1 + 2*b_q2*15:.4f}")
print(f"  At exper = 20: Δlog_wage/Δexper = {b_q1 + 2*b_q2*20:.4f}")
print("Diminishing returns: each additional year of experience raises wages less than the last.")

# Residuals plot for comparison
for name, m, c in [('Linear', m_linear, '#d62728'), ('Quadratic', m_quad, '#2ca02c')]:
    axes[1].scatter(df['exper'], m.resid, alpha=0.25, color=c, s=10, label=name)
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_xlabel('Years of Experience', fontsize=11)
axes[1].set_ylabel('Residuals', fontsize=11)
axes[1].set_title('Residuals vs. Experience\n(linear model has systematic pattern)', fontsize=10)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/03_experience_nonlinearity.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Neutral Controls: Variables That Reduce Variance Without Inducing Bias

A **neutral control** is a variable that:
- Predicts the outcome (wages) reasonably well
- Does *not* strongly predict the treatment (schooling), conditional on other controls

Adding a neutral control reduces $\sigma(\hat{\varepsilon})$ (the residual noise) without substantially reducing $\sigma(\tilde{T})$ (the residualized treatment variance). Looking at the SE formula $\sigma(\hat{\varepsilon}) / (\sigma(\tilde{T}) \cdot \sqrt{n-df})$, the numerator falls and the denominator is roughly unchanged → SE decreases.

**Example:** `KWW` (Knowledge of World of Work score) — occupational aptitude. It predicts wages independently of schooling and IQ, but is not a major driver of *how much schooling* someone pursues.

In [ ]:
BASE    = 'log_wage ~ educ + IQ + exper + I(exper**2) + tenure + married + black + south + urban'
W_KWW   = BASE + ' + KWW'
W_SIBS  = BASE + ' + sibs'
W_BOTH  = BASE + ' + KWW + sibs'

m_base  = smf.ols(BASE,   data=df).fit()
m_kww   = smf.ols(W_KWW,  data=df).fit()
m_sibs  = smf.ols(W_SIBS, data=df).fit()
m_both  = smf.ols(W_BOTH, data=df).fit()

# Check KWW: does it predict wages? Does it predict schooling?
kww_on_wage  = smf.ols('log_wage ~ KWW', data=df).fit().params['KWW']
kww_on_educ  = smf.ols('educ ~ KWW + IQ + exper + I(exper**2) + tenure + married + black + south + urban', data=df).fit().params['KWW']

print("=== KWW as a neutral control ===")
print(f"KWW → log_wage (bivariate): {kww_on_wage:.5f}   (predicts wages ✓)")
print(f"KWW → educ (partial):       {kww_on_educ:.5f}   (weak predictor of schooling ✓)")
print()
print(f"Base model — β_educ: {m_base.params['educ']:.5f}, SE: {m_base.bse['educ']:.5f}")
print(f"+ KWW      — β_educ: {m_kww.params['educ']:.5f}, SE: {m_kww.bse['educ']:.5f}  ← SE decreased")
print(f"Δ SE (KWW): {(m_kww.bse['educ'] - m_base.bse['educ']):.5f} ({(m_kww.bse['educ']/m_base.bse['educ'] - 1)*100:.1f}% change)")

## 4. Noise-Inducing Controls: Variables That Inflate SE by Absorbing Treatment Variance

A **noise-inducing control** is a variable that:
- Strongly predicts the treatment (schooling)
- Has a weak direct effect on wages

Including it reduces $\sigma(\tilde{T})$ (schooling is more predictable, so less residual variation remains) without a compensating reduction in $\sigma(\hat{\varepsilon})$. The SE formula inflates: the denominator falls, but the numerator doesn't.

**Example:** `sibs` (number of siblings). Family size is a well-known predictor of educational attainment (resource dilution hypothesis). But once you control for IQ, experience, and demographics, siblings have a weak direct effect on wages — they influence wages primarily *through* schooling, not independently of it.

In [ ]:
# Check sibs: does it predict schooling? Does it predict wages (conditional on schooling)?
sibs_on_educ = smf.ols('educ ~ sibs + IQ + exper + I(exper**2) + tenure + married + black + south + urban', data=df).fit()
sibs_on_wage_direct = smf.ols('log_wage ~ sibs + IQ + exper + I(exper**2) + tenure + married + black + south + urban', data=df).fit()

print("=== sibs as a noise-inducing control ===")
print(f"sibs → educ (partial, t-stat):           {sibs_on_educ.tvalues['sibs']:.2f}   (strong predictor of schooling)")
print(f"sibs → log_wage (direct, no educ, t-stat): {sibs_on_wage_direct.tvalues['sibs']:.2f}   (weak direct effect on wages)")
print()
print(f"Base model — β_educ: {m_base.params['educ']:.5f}, SE: {m_base.bse['educ']:.5f}")
print(f"+ sibs     — β_educ: {m_sibs.params['educ']:.5f}, SE: {m_sibs.bse['educ']:.5f}  ← SE increased")
print(f"Δ SE (sibs): {(m_sibs.bse['educ'] - m_base.bse['educ']):.5f} ({(m_sibs.bse['educ']/m_base.bse['educ'] - 1)*100:.1f}% change)")
print()

# Why: adding sibs reduces sigma(T_tilde)
debias_base = smf.ols('educ ~ IQ + exper + I(exper**2) + tenure + married + black + south + urban', data=df).fit()
debias_sibs = smf.ols('educ ~ sibs + IQ + exper + I(exper**2) + tenure + married + black + south + urban', data=df).fit()

print(f"σ(T̃) without sibs: {debias_base.resid.std():.4f}")
print(f"σ(T̃) with sibs:    {debias_sibs.resid.std():.4f}")
print(f"Adding sibs absorbs {(1 - debias_sibs.resid.std()/debias_base.resid.std())*100:.1f}% of remaining education variance.")
print("Less treatment variation → less power to detect the schooling effect → wider CI.")

## 5. The Bias-Variance Trade-Off in Practice

The tension: every confounder you include removes bias but also reduces $\sigma(\tilde{T})$. Strong predictors of schooling that have weak direct effects on wages create an especially bad trade-off — they cost you variance without much bias reduction.

The practical question — *how weak does a confounder's direct effect need to be before you're better off excluding it?* — has no universal answer. What we can do is show the trade-off empirically.

In [ ]:
spec_labels = [
    'Naive (educ only)',
    '+ Experience quadratic',
    '+ Tenure & Demographics',
    '+ IQ (confound)',
    '+ KWW (neutral)',
    '+ sibs (noise-inducing)',
    '+ KWW + sibs (both)',
]
spec_formulas = [
    'log_wage ~ educ',
    'log_wage ~ educ + exper + I(exper**2)',
    'log_wage ~ educ + exper + I(exper**2) + tenure + married + black + south + urban',
    'log_wage ~ educ + IQ + exper + I(exper**2) + tenure + married + black + south + urban',
    'log_wage ~ educ + IQ + KWW + exper + I(exper**2) + tenure + married + black + south + urban',
    'log_wage ~ educ + IQ + sibs + exper + I(exper**2) + tenure + married + black + south + urban',
    'log_wage ~ educ + IQ + KWW + sibs + exper + I(exper**2) + tenure + married + black + south + urban',
]

fitted = {lbl: smf.ols(f, data=df).fit() for lbl, f in zip(spec_labels, spec_formulas)}

table = pd.DataFrame({
    lbl: {
        'β_educ': m.params['educ'],
        'SE': m.bse['educ'],
        '95% CI width': (m.conf_int().loc['educ', 1] - m.conf_int().loc['educ', 0]),
        't-stat': m.tvalues['educ'],
        'R²': m.rsquared,
    }
    for lbl, m in fitted.items()
}).T.round(5)

print("Control selection: coefficient, precision, and R² across specifications:\n")
print(table.to_string())

In [ ]:
# Visual summary: β_educ ± 95% CI across all specifications
fig, ax = plt.subplots(figsize=(9, 5.5))

colors_map = {
    'Naive (educ only)': '#d62728',
    '+ Experience quadratic': '#7f7f7f',
    '+ Tenure & Demographics': '#7f7f7f',
    '+ IQ (confound)': '#1f77b4',
    '+ KWW (neutral)': '#2ca02c',
    '+ sibs (noise-inducing)': '#ff7f0e',
    '+ KWW + sibs (both)': '#9467bd',
}

for i, lbl in enumerate(spec_labels):
    m = fitted[lbl]
    b  = m.params['educ']
    lo = m.conf_int().loc['educ', 0]
    hi = m.conf_int().loc['educ', 1]
    c  = colors_map[lbl]
    ax.plot([lo, hi], [i, i], color=c, linewidth=2.5, alpha=0.85)
    ax.scatter([b], [i], color=c, s=75, zorder=5)

ax.axvline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.4)
ax.set_yticks(range(len(spec_labels)))
ax.set_yticklabels(spec_labels, fontsize=9)
ax.set_xlabel('Coefficient on Years of Schooling (95% CI)', fontsize=11)
ax.set_title('Bias-Variance Trade-Off in Control Selection\n'
             'IQ removes bias · KWW tightens CI · sibs widens CI', fontsize=11)
ax.invert_yaxis()

# Annotations
ax.annotate('Adding IQ: bias↓, CI shifts left', xy=(fitted['+ IQ (confound)'].params['educ'], 3),
            xytext=(0.070, 2.2), fontsize=8, color='#1f77b4',
            arrowprops=dict(arrowstyle='->', color='#1f77b4', lw=1.2))
ax.annotate('Adding KWW: CI narrows', xy=(fitted['+ KWW (neutral)'].conf_int().loc['educ', 1], 4),
            xytext=(0.070, 4.8), fontsize=8, color='#2ca02c',
            arrowprops=dict(arrowstyle='->', color='#2ca02c', lw=1.2))
ax.annotate('Adding sibs: CI widens', xy=(fitted['+ sibs (noise-inducing)'].conf_int().loc['educ', 1], 5),
            xytext=(0.070, 5.8), fontsize=8, color='#ff7f0e',
            arrowprops=dict(arrowstyle='->', color='#ff7f0e', lw=1.2))

plt.tight_layout()
plt.savefig('../outputs/03_control_selection.png', dpi=120, bbox_inches='tight')
plt.show()

print()
print("Summary:")
print(f"  IQ (confounder):       must include — removes bias. SE cost is worth it.")
print(f"  KWW (neutral):         include — reduces SE by {(fitted['+ KWW (neutral)'].bse['educ']/fitted['+ IQ (confound)'].bse['educ'] - 1)*100:.1f}% with no bias cost.")
print(f"  sibs (noise-inducing): think carefully — inflates SE by {(fitted['+ sibs (noise-inducing)'].bse['educ']/fitted['+ IQ (confound)'].bse['educ'] - 1)*100:.1f}%")
print(f"                         with minimal bias reduction (sibs' direct wage effect is weak).")

## 6. Summary

**On functional form:**
- Log wages fit the Mincerian proportional-returns framework and have better statistical properties (less skew, closer to homoskedastic).
- Experience needs a quadratic term. The linear model leaves a systematic pattern in residuals; the quadratic captures the concave experience-earnings profile with a peak around 15–17 years.

**On control selection:**

| Variable | Predicts wages? | Predicts schooling? | Effect on β_educ | Effect on SE | Include? |
|----------|----------------|---------------------|-----------------|--------------|----------|
| IQ | Yes (strongly) | Yes (strongly) | Removes upward bias | Increases | **Yes** (confounder) |
| KWW | Yes (moderate) | No (weakly) | Minimal | **Decreases** | **Yes** (neutral) |
| sibs | Weakly (via educ) | Yes (strongly) | Minimal | **Increases** | Questionable |

**The key intuition from the SE formula:** Controls that explain outcome variance tighten your estimate; controls that explain treatment variance loosen it. When both are true (a confounder), include it for bias reasons — but do so knowing the SE cost.

**Next:** Notebook 04 extends FWL to categorical controls (fixed effects) and examines the variance-weighting property of regression — the same property that causes TWFE DiD to fail with heterogeneous treatment timing.